# UMUD R50T — zespół modeli pęczków R17 (Colab, GPU)

**Jak uruchomić:** GPU → _Środowisko wykonawcze → Uruchom wszystko_. Po rozłączeniu znów _Uruchom wszystko_ — policzone obrazy
i ukończone modele zostaną pominięte, trening wznowi się od ostatniej epoki. **Zmieniaj tylko komórkę 1 (ustawienia).**
Komórka 2 zawiera spakowany kod sprawdzany sumą SHA‑256 (po edycji notebook użyje kopii z Drive).

**Po co.** FL w naszym zgłoszeniu pochodzi z jednego modelu pęczków (R17 refit) — to on daje największy błąd (FL/12 w wyniku).
Drugi model z tego samego treningu (develop) różni się od niego o ok. 5% na obraz, więc średnia z kilku modeli i widoków zmniejsza szum
bez zmiany konwencji pomiaru. Ten notebook liczy:

| etap | co robi | czas (T4 / L4) | wynik na Drive |
|---|---|---|---|
| **I** | develop + refit, trzy widoki każdego obrazu: zwykły, z odbiciem lustrzanym, z poprawionymi proporcjami pikseli (PNG, Siemens, Telemed); test + OSF‑35 | 20–30 min / 10–15 min | `outputs/R50T_v1_0_0/I/` |
| **T** | nowe modele R17 (inne ziarno, augmentacja proporcji jak w teście), po 17 epok; po każdym od razu jego widoki | ok. 3–4 h na model / 1,5–2 h | `outputs/R50T_v1_0_0/seeds/` |

Kontrola: widok zwykły modeli develop i refit musi odtworzyć zapisane pomiary R17C (tabela `REPRO`).
Pierwsza sesja kopiuje cache treningowy R17 z Drive (ok. 15–25 min) i zapisuje go jako jeden plik — kolejne rozpakowują go w kilka minut.

**Co mi przysłać:** tylko informację, że etap I się skończył (i potem, które ziarna są gotowe) — pliki odczytam z Drive.
Notebook niczego nie zgłasza na Kaggle. Nie uruchamiaj dwóch sesji na tym samym `RUN_NAME` jednocześnie.


In [ ]:
# =====================================================================================
# KOMÓRKA 1 — USTAWIENIA (tylko tę komórkę można zmieniać)
# =====================================================================================
RUN_NAME = 'R50T_v1_0_0'   # ten sam katalog = wznowienie
SEEDS = [1, 2]             # nowe modele R17 do wytrenowania (kolejno)
ASPECT_AUG = True          # augmentacja proporcji pikseli w treningu nowych modeli
RUN_INFER = True           # etap I: widoki develop + refit (+ ukończone nowe modele)
RUN_TRAIN = True           # etap T: trening nowych modeli, po każdym jego widoki
CHECK_IMAGE_SHA = True


In [ ]:
# =====================================================================================
# KOMÓRKA 2 — KOD. NIE EDYTUJ TEJ KOMÓRKI: zawiera spakowany kod (sprawdzany sumą SHA-256),
# każda zmiana w długim napisie poniżej go uszkodzi. Ustawienia są w komórce 1.
# =====================================================================================
import os, sys, json, time, base64, zlib, hashlib, importlib.util, subprocess, shutil
from pathlib import Path
T_START = time.time()
FIXTURE = os.environ.get('R50T_FIXTURE_ROOT')          # wyłącznie testy lokalne
if FIXTURE:
    DRIVE_ROOT = Path(FIXTURE) / 'MyDrive'; LOCAL = Path(FIXTURE) / 'content'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive'); LOCAL = Path('/content')
PROJECT = DRIVE_ROOT / 'UMUD_Challenge'
RUN = PROJECT / 'outputs' / RUN_NAME; RUN.mkdir(parents=True, exist_ok=True)
SRC = LOCAL / 'r50t_src'
PACKAGE_SHA = 'f5241ba92d12993fc2503c5835ef63a4d62a2423ca52e4bd0e2cbf2f3efde1be'
PAYLOAD = '...'

def _unpack(b64):
    raw = zlib.decompress(base64.b64decode(b64))
    if hashlib.sha256(raw).hexdigest() != PACKAGE_SHA:
        raise ValueError('SHA-256 kodu się nie zgadza')
    return raw

_backup = RUN / 'code' / f'r50t_{PACKAGE_SHA[:16]}.b64'
try:
    raw = _unpack(PAYLOAD)
    if not _backup.exists():
        _backup.parent.mkdir(parents=True, exist_ok=True); _backup.write_text(PAYLOAD)
except Exception as _e:
    if _backup.exists():
        raw = _unpack(_backup.read_text())
        print('UWAGA: kod w tej komórce był uszkodzony — użyłem nienaruszonej kopii z Drive:', _backup)
    else:
        raise RuntimeError('Kod w tej komórce jest uszkodzony (zwykle po edycji). Wczytaj świeży plik notebooka '
                           'i zmieniaj tylko komórkę 1 z ustawieniami.') from _e
for rel, content in json.loads(raw).items():
    p = SRC / rel; p.parent.mkdir(parents=True, exist_ok=True); p.write_text(content)
for module, pip_name in [('cv2', 'opencv-python-headless'), ('skimage', 'scikit-image'), ('timm', 'timm')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])
sys.path.insert(0, str(SRC))
import numpy as np, pandas as pd, torch, cv2
import importlib as _il
import umud_core, umud_model, umud_audit, r50_ens, r50_train
for _m in (umud_core, umud_model, umud_audit, r50_ens, r50_train):   # po ponownym uruchomieniu: zawsze świeży kod
    _il.reload(_m)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu' and not FIXTURE:
    raise RuntimeError('Brak GPU. Środowisko wykonawcze → Zmień typ środowiska wykonawczego → GPU, potem Uruchom wszystko.')
GPU = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu'

def log(*a):
    msg = ' '.join(str(x) for x in a); print(msg, flush=True)
    with open(RUN / 'LOG.txt', 'a') as fh:
        fh.write(time.strftime('%Y-%m-%d %H:%M:%S ') + msg + '\n')

def show(df, n=40):
    print(df.head(n).to_string(index=False), flush=True)

def cuda_gc():
    import gc
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

log('R50T start | GPU:', GPU, '| torch', torch.__version__, '| kod', PACKAGE_SHA[:12], '| RUN', RUN)


In [ ]:
# =====================================================================================
# KOMÓRKA 3 — DANE I CHECKPOINTY (kontrola sum)
# =====================================================================================
ASSETS = SRC / 'assets'
R17RUN = PROJECT / 'outputs' / 'R17_v1_0_0'
R12B_CKPT = DRIVE_ROOT / 'UMUD_R12B' / 'best_dinov2_frozen.pt'
R12B_SHA = os.environ.get('R50T_R12B_SHA', '21ec34d0f5dd7664ecab0ca608f976fdf9d2a161e2af2873c21d513cd5ae19ef')
SIDE = int(os.environ.get('R50T_SIDE', 768))
test = r50_ens.load_manifest(Path(os.environ.get('R50T_TEST_MANIFEST', ASSETS / 'test_manifest.csv')), PROJECT, CHECK_IMAGE_SHA)
osf = r50_ens.load_manifest(Path(os.environ.get('R50T_OSF_MANIFEST', ASSETS / 'osf_manifest.csv')), PROJECT, CHECK_IMAGE_SHA)
if 'group' in test.columns:          # tylko testy lokalne
    GROUPS_TEST = dict(zip(test.image_id, test.group))
else:
    GROUPS_TEST = {r.image_id: r50_ens.detect_group(r.path) for r in test.itertuples()}
GROUPS_OSF = {i: 'osf' for i in osf.image_id}
log('obrazy: test', len(test), '| OSF', len(osf), '| źródła testu:', pd.Series(GROUPS_TEST).value_counts().to_dict())
assert umud_core.sha256(R12B_CKPT) == R12B_SHA, 'Inny checkpoint rozcięgien R12B'

BASE = {}
for kind in ('develop', 'refit'):
    sel = json.loads((R17RUN / kind / 'selection.json').read_text())
    ck = R17RUN / kind / 'best.pt'
    assert umud_core.sha256(ck) == sel['checkpoint_sha256'], f'R17 {kind}: checkpoint różni się od selection.json'
    BASE[kind] = (ck, float(sel['threshold']), sel['checkpoint_sha256'])
log('checkpointy R17 develop/refit i R12B zgodne; progi:', {k: v[1] for k, v in BASE.items()})
OUTI = RUN / 'I'; OUTI.mkdir(exist_ok=True)
SEEDS_DIR = RUN / 'seeds'; SEEDS_DIR.mkdir(exist_ok=True)
TRUTH = r50_ens.osf_truth(ASSETS / 'osf35_full.csv')
SAVED = {'refit': pd.read_csv(ASSETS / 'r17c_refit_test.csv'), 'develop': pd.read_csv(ASSETS / 'r17c_develop_test.csv')}

def seed_specs():
    out = {}
    for d in sorted(SEEDS_DIR.glob('seed_*/DONE.json')):
        info = json.loads(d.read_text()); ck = d.parent / 'refit' / 'best.pt'
        assert umud_core.sha256(ck) == info['checkpoint_sha256'], f'{ck}: suma się nie zgadza'
        out[f"seed{info['seed']}"] = (ck, float(info['threshold']), info['checkpoint_sha256'])
    return out


## Etap I — widoki modeli R17 (develop, refit i ukończone nowe modele)

Dla każdego obrazu: widok zwykły (jak R17C), z odbiciem lustrzanym i z poprawionymi proporcjami pikseli.
Tabele: `REPRO` (odtworzenie R17C), `OSF_TABLE` (MAE po odjęciu stałej na OSF‑35 dla każdego modelu, widoku i średnich).


In [ ]:
# =====================================================================================
# KOMÓRKA 4 — ETAP I
# =====================================================================================
def run_inference(tag):
    specs = {**BASE, **seed_specs()}
    models = {k: umud_model.load_model(v[0], DEVICE)[0] for k, v in specs.items()}
    thr = {k: v[1] for k, v in specs.items()}
    msig = {k: {'sha': v[2], 'thr': v[1]} for k, v in specs.items()}
    apo = umud_model.load_legacy_apo(R12B_CKPT, DEVICE)
    sig = {'side': SIDE, 'r12b': R12B_SHA, 'apo_thr': 0.5}
    t0 = time.time()
    dt = r50_ens.run_views('I_test', test, GROUPS_TEST, models, thr, apo, SIDE, DEVICE, OUTI / 'test', sig, msig, log)
    do = r50_ens.run_views('I_osf', osf, GROUPS_OSF, models, thr, apo, SIDE, DEVICE, OUTI / 'osf', sig, msig, log)
    del models, apo; cuda_gc()
    rep = pd.concat([r50_ens.repro_table(dt, SAVED[m], m) for m in ('refit', 'develop')], ignore_index=True)
    rep.to_csv(OUTI / 'REPRO.csv', index=False)
    tab = r50_ens.osf_table(do, TRUTH, list(specs)); tab.to_csv(OUTI / 'OSF_TABLE.csv', index=False)
    r50_ens.wide(dt).to_csv(RUN / 'R50T_LATEST_test.csv', index=False)
    r50_ens.wide(do).to_csv(RUN / 'R50T_LATEST_osf.csv', index=False)
    umud_core.atomic_json(OUTI / 'I_SUMMARY.json', r50_ens.clean_json({
        'tag': tag, 'models': list(specs), 'gpu': GPU, 'minutes': round((time.time() - t0) / 60, 1),
        'test_rows': len(dt), 'osf_rows': len(do),
        'status_counts': dt[dt.view == 'plain'].groupby('model').status.value_counts().to_dict().__repr__(),
        'repro': rep.to_dict('records'), 'osf_best': tab.head(12).to_dict('records')}))
    print('\nREPRO (widok zwykły vs zapisane R17C):'); show(rep)
    print('\nOSF-35 (MAE po odjęciu stałej; najlepsze FL na górze):'); show(tab, 30)
    log(f'ETAP I GOTOWY ({tag}): modele {list(specs)} ->', RUN / 'R50T_LATEST_test.csv')

if RUN_INFER:
    run_inference('start')


## Etap T — nowe modele R17 (wznawialny)

Tryb refit: wszystkie dane treningowe, 17 epok (liczba wybrana w R17 develop), inne ziarno, augmentacja proporcji pikseli.
Checkpoint co epokę; po rozłączeniu uruchom wszystko ponownie. Po każdym ukończonym modelu etap I liczy tylko jego widoki.


In [ ]:
# =====================================================================================
# KOMÓRKA 5 — ETAP T
# =====================================================================================
if RUN_TRAIN and SEEDS:
    samples = pd.read_csv(R17RUN / 'samples.csv', dtype={'sample_id': str})
    if FIXTURE and os.environ.get('R50T_SAMPLES_LIMIT'):
        samples = samples.head(int(os.environ['R50T_SAMPLES_LIMIT']))
    base_cfg = json.loads((R17RUN / 'refit' / 'config.json').read_text())['config']
    if FIXTURE:
        base_cfg.update(json.loads(os.environ.get('R50T_TRAIN_OVERRIDES', '{}')))
    local = r50_train.sync_cache(samples, R17RUN / 'cache', RUN / 'cache_r17.tar', LOCAL / 'r50t_r17_cache', log=log)
    csha = r50_train.code_sha([SRC / 'umud_core.py', SRC / 'umud_model.py', SRC / 'umud_audit.py'])
    log('trening nowych modeli R17:', SEEDS, '| augmentacja proporcji:', ASPECT_AUG, '| próbki:', len(samples))
    for seed in SEEDS:
        rd = SEEDS_DIR / f'seed_{seed}'
        if (rd / 'DONE.json').exists():
            log(f'ziarno {seed}: już gotowe'); continue
        try:
            r50_train.train_seed(samples, local, rd, base_cfg, seed, R17RUN / 'develop' / 'selection.json', ASPECT_AUG, csha, log=log)
        except RuntimeError as e:
            if 'SESSION BUDGET' in str(e):
                log('Limit czasu w tej sesji — checkpoint zapisany. Uruchom wszystko ponownie.'); break
            raise
        cuda_gc()
        if RUN_INFER:
            run_inference(f'po ziarnie {seed}')
    log('Ukończone ziarna:', sorted(seed_specs()))


## Podsumowanie

Pliki do analizy: `R50T_LATEST_test.csv`, `R50T_LATEST_osf.csv`, `I/OSF_TABLE.csv`, `I/REPRO.csv`.


In [ ]:
# =====================================================================================
# KOMÓRKA 6 — PODSUMOWANIE
# =====================================================================================
files = sorted(str(p.relative_to(RUN)) for p in RUN.rglob('*') if p.suffix in ('.csv', '.json') and 'cache' not in str(p))
print('\n'.join(files))
log(f'GOTOWE. Modele w zespole: {["develop", "refit"] + sorted(seed_specs())}. Daj znać — pliki odczytam z Drive ({RUN}).')
